In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
cd/content/drive/MyDrive/mc-med_data/physionet.org/files/mc-med/1.0.0

/content/drive/MyDrive/mc-med_data/physionet.org/files/mc-med/1.0.0


In [3]:
cd data

/content/drive/MyDrive/mc-med_data/physionet.org/files/mc-med/1.0.0/data


In [4]:
import pandas as pd
import numpy as np
import re
from datetime import timedelta

In [10]:
def split_clean_icd(cell):
    """
    Splits a cell of ICD codes into a list, removes leading/trailing spaces,
    and converts to uppercase. Periods within the codes are preserved.
    """
    if pd.isna(cell):
        return []
    parts = re.split(r'[\s;]+', str(cell))
    return [p.strip().upper() for p in parts if p.strip()]

def adjust_time_by_subtracting_year(time_series):
    s = time_series.astype("string")
    year = pd.to_numeric(s.str[:4], errors='coerce')
    month_day_time = s.str[4:]
    new_year = year - 200
    new_time_str = new_year.astype('Int64').astype('string') + month_day_time
    new_time_str = new_time_str.where(s.notna(), pd.NA)
    return pd.to_datetime(new_time_str, errors='coerce').dt.tz_localize(None)

In [11]:
df_visits = pd.read_csv('visits.csv')
df_meds = pd.read_csv('meds.csv')
df_pmh = pd.read_csv('pmh.csv')

df_visits['Arrival_time'] = adjust_time_by_subtracting_year(df_visits['Arrival_time'])
df_meds[['Start_date','End_date']] = df_meds[['Start_date','End_date']].apply(adjust_time_by_subtracting_year)
df_pmh['Noted_date'] = adjust_time_by_subtracting_year(df_pmh['Noted_date'])

# df_visits.dropna(subset=['Arrival_time'], inplace=True)
df_visits['Admit_service'] = df_visits['Admit_service'].astype(str).fillna('')
df_visits = df_visits.sort_values(by=['MRN', 'Arrival_time']).reset_index(drop=True)

df_meds.set_index('MRN', inplace=True)

# df_pmh.dropna(subset=['Noted_date'], inplace=True)
df_pmh['Code'] = df_pmh['Code'].astype(str)
df_pmh['Code'] = df_pmh['Code'].str.replace('.', '', regex=False)
df_pmh = df_pmh.sort_values(by=['MRN', 'Noted_date'])

charlson_codes_v9 = {
    'cci_MI': ['410', '412'],
    'cci_CHF': ['39891', '40201', '40211', '40291', '40401', '40403', '40411', '40413', '40491', '40493', '4254', '4255', '4256', '4257', '4258', '4259', '428'],
    'cci_PVD': ['4431', '4432', '4433', '4434', '4435', '4436', '4437', '4438', '4439', '0930', '4373', '4471', '5571', '5579', 'V434', '440', '441'],
    'cci_Stroke': ['36234', '430', '431', '432', '433', '434', '435', '436', '437', '438'],
    'cci_Dementia': ['2941', '3312', '290'],
    'cci_Pulmonary': ['4168', '4169', '5064', '5081', '5088', '490', '491', '492', '493', '494', '495', '496', '497', '498', '499', '500', '501', '502', '503', '504', '505'],
    'cci_Rheumatic': ['4465', '7100', '7101', '7102', '7103', '7104', '7140', '7141', '7142', '7148', '725'],
    'cci_PUD': ['531', '532', '533', '534'],
    'cci_Liver1': ['07022', '07023', '07032', '07033', '07044', '07054', '0706', '0709', '5733', '5734', '5738', '5739', 'V427', '570', '571'],
    'cci_DM1': ['2500', '2501', '2502', '2503', '2508', '2509'],
    'cci_DM2': ['2504', '2505', '2506', '2507'],
    'cci_Paralysis': ['3340', '3341', '3342', '3343', '3344', '3345', '3346', '3349', '342', '343'],
    'cci_Renal': ['40301', '40311', '40391', '40402', '40403', '40412', '40413', '40492', '40493', '5830', '5831', '5832', '5833', '5834', '5835', '5836', '5837', '5880', 'V420', 'V451', '585', '586', 'V56', '582'],
    'cci_Cancer1': ['2386', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170', '171', '172', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '190', '191', '192', '193', '194', '200', '201', '202', '203', '204', '205', '206', '207', '208'],
    'cci_Liver2': ['4560', '4561', '4562', '5722', '5723', '5724', '5725', '5726', '5727', '5728'],
    'cci_Cancer2': ['196', '197', '198', '199'],
    'cci_HIV': ['042', '043', '044']
}
charlson_codes_v10 = {
    'cci_MI': ['I252', 'I21', 'I22'],
    'cci_CHF': ['I099', 'I110', 'I132', 'I255', 'I420', 'I425', 'I426', 'I427', 'I428', 'I429', 'P290', 'I50', 'I43'],
    'cci_PVD': ['I731', 'I738', 'I739', 'I771', 'I790', 'I792', 'K551', 'K558', 'K559', 'Z958', 'Z959', 'I70', 'I71'],
    'cci_Stroke': ['H340', 'G45', 'G46', 'I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69'],
    'cci_Dementia': ['F051', 'G311', 'F00', 'F01', 'F02', 'F03', 'G30'],
    'cci_Pulmonary': ['I278', 'I279', 'J684', 'J701', 'J703', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47', 'J60', 'J61', 'J62', 'J63', 'J64', 'J65', 'J66', 'J67'],
    'cci_Rheumatic': ['M315', 'M351', 'M353', 'M360', 'M05', 'M06', 'M32', 'M33', 'M34'],
    'cci_PUD': ['K25', 'K26', 'K27', 'K28'],
    'cci_Liver1': ['K700', 'K701', 'K702', 'K703', 'K709', 'K713', 'K714', 'K715', 'K717', 'K760', 'K762', 'K763', 'K764', 'K768', 'K769', 'Z944', 'B18', 'K73', 'K74'],
    'cci_DM1': ['E100', 'E101', 'E106', 'E108', 'E109', 'E110', 'E111', 'E116', 'E118', 'E119', 'E120', 'E121', 'E126', 'E128', 'E129', 'E130', 'E131', 'E136', 'E138', 'E139', 'E140', 'E141', 'E146', 'E148', 'E149'],
    'cci_DM2': ['E102', 'E103', 'E104', 'E105', 'E107', 'E112', 'E113', 'E114', 'E115', 'E117', 'E122', 'E123', 'E124', 'E125', 'E127', 'E132', 'E133', 'E134', 'E135', 'E137', 'E142', 'E143', 'E144', 'E145', 'E147'],
    'cci_Paralysis': ['G041', 'G114', 'G801', 'G802', 'G830', 'G831', 'G832', 'G833', 'G834', 'G839', 'G81', 'G82'],
    'cci_Renal': ['I120', 'I131', 'N032', 'N033', 'N034', 'N035', 'N036', 'N037', 'N052', 'N053', 'N054', 'N055', 'N056', 'N057', 'N250', 'Z490', 'Z491', 'Z492', 'Z940', 'Z992', 'N18', 'N19'],
    'cci_Cancer1': ['C00', 'C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C43', 'C88', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26', 'C30', 'C31', 'C32', 'C33', 'C34', 'C37', 'C38', 'C39', 'C40', 'C41', 'C45', 'C46', 'C47', 'C48', 'C49', 'C50', 'C51', 'C52', 'C53', 'C54', 'C55', 'C56', 'C57', 'C58', 'C60', 'C61', 'C62', 'C63', 'C64', 'C65', 'C66', 'C67', 'C68', 'C69', 'C70', 'C71', 'C72', 'C73', 'C74', 'C75', 'C76', 'C81', 'C82', 'C83', 'C84', 'C85', 'C90', 'C91', 'C92', 'C93', 'C94', 'C95', 'C96', 'C97'],
    'cci_Liver2': ['I850', 'I859', 'I864', 'I982', 'K704', 'K711', 'K721', 'K729', 'K765', 'K766', 'K767'],
    'cci_Cancer2': ['C77', 'C78', 'C79', 'C80'],
    'cci_HIV': ['B20', 'B21', 'B22', 'B24']
}
elixhauser_codes_v9 = {
    'eci_HIV': ['042', '043', '044'],
    'eci_Alcohol': ['2652', '2911', '2913', '2915', '2916', '2917', '2918', '2919', '3030', '3039', '3050', '3575', '4255', '5353', '5710', '5711', '5712', '5713', 'V113', '980'],
    'eci_BloodLoss': ['2800'],
    'eci_Arrhythmia': ['4260', '42613', '4267', '4269', '42610', '42612', '4270', '4271', '4272', '4273', '4274', '4276', '4277', '4278', '4279', '7850', '99601', '99604', 'V450', 'V533'],
    'eci_Pulmonary': ['4168', '4169', '5064', '5081', '5088', '490', '491', '492', '493', '494', '495', '496', '497', '498', '499', '500', '501', '502', '503', '504', '505'],
    'eci_Coagulopathy': ['2871', '2873', '2874', '2875', '286'],
    'eci_CHF': ['39891', '40201', '40211', '40291', '40401', '40403', '40411', '40413', '40491', '40493', '4254', '4255', '4256', '4257', '4258', '4259', '428'],
    'eci_Anemia': ['2801', '2802', '2803', '2804', '2805', '2806', '2807', '2808', '2809', '281'],
    'eci_Depression': ['2962', '2963', '2965', '3004', '311', '309'],
    'eci_DM2': ['2504', '2505', '2506', '2507', '2508', '2509'],
    'eci_DM1': ['2500', '2501', '2502', '2503'],
    'eci_Drugs': ['3052', '3053', '3054', '3055', '3056', '3057', '3058', '3059', 'V6542', '292', '304'],
    'eci_FluidsLytes': ['2536', '276'],
    'eci_HTN2': ['402', '403', '404', '405'],
    'eci_HTN1': ['401'],
    'eci_Hypothyroid': ['2409', '2461', '2468', '243', '244'],
    'eci_Liver': ['07022', '07023', '07032', '07033', '07044', '07054', '0706', '0709', '4560', '4561', '4562', '5722', '5723', '5724', '5725', '5726', '5727', '5728', '5733', '5734', '5738', '5739', 'V427', '570', '571'],
    'eci_Lymphoma': ['2030', '2386', '200', '201', '202'],
    'eci_Tumor2': ['196', '197', '198', '199'], # Metastatic Cancer
    'eci_Obesity': ['2780'],
    'eci_NeuroOther': ['3319', '3320', '3321', '3334', '3335', '33392', '3362', '3481', '3483', '7803', '7843', '334', '335', '340', '341', '345'],
    'eci_Paralysis': ['3440', '3441', '3442', '3443', '3444', '3445', '3446', '3341', '3449', '342', '343'],
    'eci_PUD': ['5317', '5319', '5327', '5329', '5337', '5339', '5347', '5349'],
    'eci_PVD': ['0930', '4373', '4431', '4432', '4433', '4434', '4435', '4436', '4437', '4438', '4439', '4471', '5571', '5579', 'V434', '440', '441'],
    'eci_Psychoses': ['2938', '29604', '29614', '29644', '29654', '295', '297', '298'],
    'eci_PHTN': ['4150', '4151', '4170', '4178', '4179', '416'],
    'eci_Renal': ['40301', '40311', '40391', '40402', '40403', '40493', '5880', 'V420', '585', '586', 'V56', 'V451'],
    'eci_Rheumatic': ['7010', '7100', '7101', '7102', '7103', '7104', '7108', '7109', '7112', '7193', '7285', '72889', '72930', '446', '714', '720', '725'],
    'eci_Tumor1': ['140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170', '171', '172', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195'], # Solid tumor without metastasis
    'eci_Valvular': ['0932', '7463', '7464', '7465', '7466', 'V422', 'V433', '394', '395', '396', '397', '424'],
    'eci_WeightLoss': ['7832', '7994', '260', '261', '262', '263']
}
elixhauser_codes_v10 = {
    'eci_HIV': ['B20', 'B21', 'B22', 'B24'],
    'eci_Alcohol': ['F10', 'E52', 'G621', 'I426', 'K292', 'K700', 'K703', 'K709', 'T51', 'Z502', 'Z714', 'Z721'],
    'eci_BloodLoss': ['D500'],
    'eci_Arrhythmia': ['I441', 'I442', 'I443', 'I456', 'I459', 'R000', 'R001', 'R008', 'T821', 'Z450', 'Z950', 'I47', 'I48', 'I49'],
    'eci_Pulmonary': ['I278', 'I279', 'J684', 'J701', 'J703', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47', 'J60', 'J61', 'J62', 'J63', 'J64', 'J65', 'J66', 'J67'],
    'eci_Coagulopathy': ['D691', 'D693', 'D694', 'D695', 'D696', 'D65', 'D66', 'D67', 'D68'],
    'eci_CHF': ['I099', 'I110', 'I130', 'I132', 'I255', 'I420', 'I425', 'I426', 'I427', 'I428', 'I429', 'P290', 'I43', 'I50'],
    'eci_Anemia': ['D508', 'D509', 'D51', 'D52', 'D53'],
    'eci_Depression': ['F204', 'F313', 'F314', 'F315', 'F32', 'F33', 'F341', 'F412', 'F432'],
    'eci_DM2': ['E102', 'E103', 'E104', 'E105', 'E106', 'E107', 'E108', 'E112', 'E113', 'E114', 'E115', 'E116', 'E117', 'E118', 'E122', 'E123', 'E124', 'E125', 'E126', 'E127', 'E128', 'E132', 'E133', 'E134', 'E135', 'E136', 'E137', 'E138', 'E142', 'E143', 'E144', 'E145', 'E146', 'E147', 'E148'],
    'eci_DM1': ['E100', 'E101', 'E109', 'E110', 'E111', 'E119', 'E120', 'E121', 'E129', 'E130', 'E131', 'E139', 'E140', 'E141', 'E149'],
    'eci_Drugs': ['F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F18', 'F19', 'Z715', 'Z722'],
    'eci_FluidsLytes': ['E86', 'E87'],
    'eci_HTN2': ['I11', 'I12', 'I13', 'I15'],
    'eci_HTN1': ['I10'],
    'eci_Hypothyroid': ['E890', 'E00', 'E01', 'E02', 'E03'],
    'eci_Liver': ['I864', 'I982', 'K711', 'K713', 'K714', 'K715', 'K717', 'K760', 'Z944', 'K762', 'K763', 'K764', 'K765', 'K766', 'K767', 'K768', 'K769', 'B18', 'I85', 'K70', 'K72', 'K73', 'K74'],
    'eci_Lymphoma': ['C900', 'C902', 'C81', 'C82', 'C83', 'C84', 'C85', 'C88', 'C96'],
    'eci_Tumor2': ['C77', 'C78', 'C79', 'C80'], # Metastatic Cancer
    'eci_Obesity': ['E66'],
    'eci_NeuroOther': ['G254', 'G255', 'G312', 'G318', 'G319', 'G931', 'G934', 'R470', 'G10', 'G11', 'G12', 'G13', 'G20', 'G21', 'G22', 'G32', 'G35', 'G36', 'G37', 'G40', 'G41', 'R56'],
    'eci_Paralysis': ['G041', 'G114', 'G801', 'G802', 'G830', 'G831', 'G832', 'G833', 'G834', 'G839', 'G81', 'G82'],
    'eci_PUD': ['K257', 'K259', 'K267', 'K269', 'K277', 'K279', 'K287', 'K289'],
    'eci_PVD': ['I731', 'I738', 'I739', 'I771', 'I790', 'I792', 'IK551', 'K558', 'K559', 'Z958', 'Z959', 'I70', 'I71'],
    'eci_Psychoses': ['F20', 'F22', 'F23', 'F24', 'F25', 'F28', 'F29', 'F302', 'F312', 'F315'],
    'eci_PHTN': ['I280', 'I288', 'I289', 'I26', 'I27'],
    'eci_Renal': ['I120', 'I131', 'N250', 'Z490', 'Z491', 'Z492', 'Z940', 'Z992', 'N18', 'N19'],
    'eci_Rheumatic': ['L940', 'L941', 'L943', 'M120', 'M123', 'M310', 'M311', 'M312', 'M313', 'M461', 'M468', 'M469', 'M05', 'M06', 'M08', 'M32', 'M33', 'M34', 'M35', 'M45'],
    'eci_Tumor1': ['C00', 'C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26', 'C30', 'C31', 'C32', 'C33', 'C34','C37', 'C38', 'C39', 'C40', 'C41', 'C43', 'C45', 'C46', 'C47', 'C48', 'C49', 'C50', 'C51', 'C52', 'C53', 'C54', 'C55', 'C56', 'C57', 'C58','C60', 'C61', 'C62', 'C63', 'C64', 'C65', 'C66', 'C67', 'C68', 'C69', 'C70', 'C71', 'C72', 'C73', 'C74', 'C75', 'C76','C97'], # Solid tumor without metastasis
    'eci_Valvular': ['A520', 'I091', 'I098', 'Q230', 'Q231', 'Q232', 'Q233', 'Z952', 'Z953', 'Z954', 'I05', 'I06', 'I07', 'I08', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39'],
    'eci_WeightLoss': ['R634', 'R64', 'E40', 'E41', 'E42', 'E43', 'E44', 'E45', 'E46']
}


In [12]:
comorbidity_codes = {}
for condition, codes in charlson_codes_v9.items():
    comorbidity_codes[condition] = {'icd9': {'starts': codes}}
for condition, codes in charlson_codes_v10.items():
    comorbidity_codes[condition]['icd10'] = {'starts': codes}

for condition, codes in elixhauser_codes_v9.items():
    comorbidity_codes[condition] = {'icd9': {'starts': codes}}
for condition, codes in elixhauser_codes_v10.items():
    comorbidity_codes[condition]['icd10'] = {'starts': codes}

In [13]:
def calculate_full_history(row):
    current_mrn = row['MRN']
    current_csn = row['CSN']
    current_arrival_time = row['Arrival_time']

    # --- calculate medical history ---
    past_visits_mask = (
        (df_visits['MRN'] == current_mrn) &
        (df_visits['CSN'] != current_csn) &
        (df_visits['Arrival_time'] <= current_arrival_time)
    )
    past_visits = df_visits.loc[past_visits_mask]

    n_ed_30d, n_ed_90d, n_ed_365d = 0, 0, 0
    n_hosp_30d, n_hosp_90d, n_hosp_365d = 0, 0, 0
    n_icu_30d, n_icu_90d, n_icu_365d = 0, 0, 0

    if not past_visits.empty:
        time_diffs = (current_arrival_time - past_visits['Arrival_time']).dt.days

        # Count all past ED visits
        n_ed_30d = (time_diffs <= 30).sum()
        n_ed_90d = (time_diffs <= 90).sum()
        n_ed_365d = (time_diffs <= 365).sum()

        # Past hospitalizations ("Inpatient")
        past_admissions = past_visits[past_visits["ED_dispo"] == "Inpatient"]
        if not past_admissions.empty:
            hosp_time_diffs = (current_arrival_time - past_admissions['Arrival_time']).dt.days
            n_hosp_30d = (hosp_time_diffs <= 30).sum()
            n_hosp_90d = (hosp_time_diffs <= 90).sum()
            n_hosp_365d = (hosp_time_diffs <= 365).sum()

        # Past ICU visits
        past_icu_visits = past_visits[past_visits["ED_dispo"] == "ICU"]
        if not past_icu_visits.empty:
            icu_time_diffs = (current_arrival_time - past_icu_visits['Arrival_time']).dt.days
            n_icu_30d = (icu_time_diffs <= 30).sum()
            n_icu_90d = (icu_time_diffs <= 90).sum()
            n_icu_365d = (icu_time_diffs <= 365).sum()

    # --- calculate medication history ---
    n_med, n_medrecon = 0, 0
    if current_mrn in df_meds.index:
        patient_meds_data = df_meds.loc[current_mrn]
        if isinstance(patient_meds_data, pd.Series):
            patient_meds = patient_meds_data.to_frame().T
        else:
            patient_meds = patient_meds_data

        five_years_prior = current_arrival_time - timedelta(days=5*365)
        meds_in_5y_mask = (patient_meds['Start_date'] >= five_years_prior) & \
                          (patient_meds['Start_date'] <= current_arrival_time)
        meds_in_5y = patient_meds[meds_in_5y_mask]
        n_med = meds_in_5y['Generic_name'].nunique()
        starts_in_5y = (patient_meds['Start_date'] >= five_years_prior) & \
                       (patient_meds['Start_date'] <= current_arrival_time)
        ends_in_5y = (patient_meds['End_date'] >= five_years_prior) & \
                     (patient_meds['End_date'] <= current_arrival_time)
        n_medrecon = starts_in_5y.sum() + ends_in_5y.sum()

    # --- calculate comorbidity ---
    comorbidity_results = {key: 0 for key in comorbidity_codes.keys()}
    patient_diagnoses_mask = (df_pmh['MRN'] == current_mrn) & (df_pmh['Noted_date'] <= current_arrival_time)
    patient_diagnoses = df_pmh.loc[patient_diagnoses_mask]

    if not patient_diagnoses.empty:
        for condition, codes in comorbidity_codes.items():
            # ICD-9
            if 'icd9' in codes and 'starts' in codes['icd9']:
                icd9_codes = patient_diagnoses[patient_diagnoses['CodeType'] == 'Dx9']['Code']
                if not icd9_codes.empty:
                    if isinstance(codes['icd9']['starts'], list):
                        codes['icd9']['starts'] = tuple(codes['icd9']['starts'])
                    if icd9_codes.str.startswith(codes['icd9']['starts']).any():
                        comorbidity_results[condition] = 1

            # ICD-10
            if 'icd10' in codes and 'starts' in codes['icd10']:
                icd10_codes = patient_diagnoses[patient_diagnoses['CodeType'] == 'Dx10']['Code']
                if not icd10_codes.empty:
                    if isinstance(codes['icd10']['starts'], list):
                        codes['icd10']['starts'] = tuple(codes['icd10']['starts'])
                    if icd10_codes.str.startswith(codes['icd10']['starts']).any():
                        comorbidity_results[condition] = 1

    return (n_ed_30d, n_ed_90d, n_ed_365d,
            n_hosp_30d, n_hosp_90d, n_hosp_365d,
            n_icu_30d, n_icu_90d, n_icu_365d,
            n_med, n_medrecon,
            *comorbidity_results.values())

history_column_names = [
    'n_ed_30d', 'n_ed_90d', 'n_ed_365d',
    'n_hosp_30d', 'n_hosp_90d', 'n_hosp_365d',
    'n_icu_30d', 'n_icu_90d', 'n_icu_365d',
    'n_med', 'n_medrecon'
]
comorbidity_columns = list(comorbidity_codes.keys())
all_new_columns = history_column_names + comorbidity_columns

new_features_df = df_visits.apply(calculate_full_history, axis=1, result_type='expand')
new_features_df.columns = all_new_columns

df_final = pd.concat([df_visits, new_features_df], axis=1)

In [14]:
final_columns_to_keep = ['MRN', 'CSN'] + all_new_columns
df_final_subset = df_final[final_columns_to_keep]

print(df_final_subset.head())

        MRN       CSN  n_ed_30d  n_ed_90d  n_ed_365d  n_hosp_30d  n_hosp_90d  \
0  98816154  99110673         0         0          0           0           0   
1  98816163  99514339         0         0          0           0           0   
2  98816201  99371371         0         0          0           0           0   
3  98816214  99088308         0         0          0           0           0   
4  98816238  99361890         0         0          0           0           0   

   n_hosp_365d  n_icu_30d  n_icu_90d  ...  eci_Paralysis  eci_PUD  eci_PVD  \
0            0          0          0  ...              0        0        0   
1            0          0          0  ...              0        0        0   
2            0          0          0  ...              0        0        0   
3            0          0          0  ...              0        0        0   
4            0          0          0  ...              0        0        0   

   eci_Psychoses  eci_PHTN  eci_Renal  eci_Rheumat

In [15]:
comorbidity_counts = df_final_subset[comorbidity_columns].sum()
total_visits = len(df_final_subset)
comorbidity_percentages = (comorbidity_counts / total_visits) * 100
print(comorbidity_percentages)

cci_MI               3.936309
cci_CHF              7.385226
cci_PVD              5.083414
cci_Stroke           7.052414
cci_Dementia         1.436837
cci_Pulmonary        9.491067
cci_Rheumatic        2.333911
cci_PUD              0.970562
cci_Liver1           6.275288
cci_DM1             10.590869
cci_DM2              3.505512
cci_Paralysis        0.742493
cci_Renal            7.589644
cci_Cancer1         13.736538
cci_Liver2           1.731638
cci_Cancer2          3.213245
cci_HIV              0.348017
eci_HIV              0.348017
eci_Alcohol          4.449888
eci_BloodLoss        1.501879
eci_Arrhythmia      14.072729
eci_Pulmonary        9.491067
eci_Coagulopathy     3.776661
eci_CHF              7.387760
eci_Anemia           4.255607
eci_Depression      10.063775
eci_DM2              5.624023
eci_DM1              9.059425
eci_Drugs            3.967563
eci_FluidsLytes     11.840182
eci_HTN2             0.989146
eci_HTN1            21.372640
eci_Hypothyroid      6.378342
eci_Liver 

In [16]:
print(df_final_subset)

             MRN       CSN  n_ed_30d  n_ed_90d  n_ed_365d  n_hosp_30d  \
0       98816154  99110673         0         0          0           0   
1       98816163  99514339         0         0          0           0   
2       98816201  99371371         0         0          0           0   
3       98816214  99088308         0         0          0           0   
4       98816238  99361890         0         0          0           0   
...          ...       ...       ...       ...        ...         ...   
118380  99999964  99194822         0         0          0           0   
118381  99999964  99699330         0         1          1           0   
118382  99999964  99713052         0         2          2           0   
118383  99999976  99828601         0         0          0           0   
118384  99999985  99082100         0         0          0           0   

        n_hosp_90d  n_hosp_365d  n_icu_30d  n_icu_90d  ...  eci_Paralysis  \
0                0            0          0    

In [ ]:
# To save this clean table to a file:
df_final_subset.to_csv('dx_history_dataset.csv', index=False)